# NHL: предсказать успешность команды в следующем сезоне

**Тестовое задание • 100 баллов**

Вы аналитик, которому после окончания сезона нужно оценить перспективы команды
на следующий год. Соберите таблицу с учебного сайта
[Scrape This Site - Hockey Teams](https://www.scrapethissite.com/pages/forms/),
изучите данные и постройте модель классификации.
В работе оцениваются корректность исследования и обоснованность решений.

Ориентир времени: 6-10 часов (знание хоккея не требуется).
Можно обращаться к документации и пользоваться вспомогательными инструментами.
Укажите, чем пользовались; отвечайте за правильность и понимание сданной работы.
Подтверждайте ответы результатами из своего ноутбука.

## Что сдавать

Выполненный IPynb с видимыми результатами и ответами и исходный `data/hockey_raw.csv` разместите на платформе githib.com.
Укажите имя, дату и использованные источники и инструменты в README.

## Что предсказывает модель

Одна строка признаков описывает команду в сезоне `t`.
`target=1`, если доля побед той же команды в сезоне `t+1` строго выше медианной
доли побед всех команд в сезоне `t+1`. В остальных случаях `target=0`.

Медиану считайте по полной исходной таблице будущего года до отбора пар.
В пары включайте одинаковые названия команды с разницей меток сезонов ровно в один год.
Строки без пары исключайте и учитывайте в описании данных.
Будущий `win_rate` и будущая медиана используются для построения target и анализа
готовых прогнозов. Признаки модели должны быть известны после сезона `t`.

## Схема проверки

Делите данные по году целевого сезона:

| Часть | target_year | Назначение |
|---|---|---|
| train | не позднее 2005 | Подготовка данных и обучение |
| validation | 2006-2008 | Сравнение вариантов и выбор решения |
| test | 2009-2011 | Одна итоговая оценка выбранного решения |

Основная метрика: ROC-AUC по вероятности класса 1.
На итоговом этапе также используйте F1 и accuracy при пороге 0.5.
Финальный test используйте после выбора решения на validation.

## Как оценивается работа

| Область | Баллы |
|---|---:|
| Сбор и понимание данных | 15 |
| Построение задачи и корректность проверки | 20 |
| Исследовательский анализ | 20 |
| Модели и эксперименты | 25 |
| Итоговая оценка и анализ ошибок | 15 |
| Ясность и воспроизводимость | 5 |
| **Всего** | **100** |

Ниже перечислены обязательные этапы работы. Внутри них вы самостоятельно выбираете
исследовательские вопросы, графики, признаки, обработку данных, эксперимент и способ
разбора ошибок. Оценивается качество решений, а не совпадение с эталонным маршрутом.
Сложность метода сама по себе не даёт преимущества. Корректный эксперимент может
получить полный балл и в случае отсутствия улучшения.

## Требования к сбору

Итоговая исходная таблица должна иметь столбцы `team`, `year`, `wins`, `losses`,
`ot_losses`, `win_rate`, `goals_for`, `goals_against`, `goal_diff`.
Самостоятельно исследуйте страницу и определите способ получения всей таблицы.

    - Автор: Нийоншути Мартин
    - Дата: 2026-10-02  
    - Использованные источники и инструменты: https://www.scrapethissite.com/pages/forms/

In [92]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import time
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Load IPython extension to show cell execution time
%reload_ext autotime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

time: 0 ns (started: 2026-10-02 01:40:04 +03:00)


## Раздел 1. Сбор и понимание данных: 15 баллов

### 1.1. Соберите и сохраните данные

Соберите все страницы сайта в одну таблицу с указанными девятью столбцами.
Сохраните исходную выгрузку в `data/hockey_raw.csv`. Укажите URL и дату сбора,
число обработанных страниц и число строк. Покажите, как проверили,
что сбор не ограничился первой страницей и при повторном анализе можно читать CSV.

In [93]:
# Configuration
BASE_URL = "https://www.scrapethissite.com/pages/forms/"
DATA_DIR = "data"
OUTPUT_FILE = os.path.join(DATA_DIR, "hockey_raw.csv")

# Create data directory if it doesn't exist
os.makedirs(DATA_DIR, exist_ok=True)

# Function to scrape a single page
def scrape_page(page_num=1):
    if page_num == 1:
        url = BASE_URL
    else:
        url = f"{BASE_URL}?page_num={page_num}"
    
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Find the table
    table = soup.find('table', class_='table')
    if not table:
        return []
    
    rows = []
    for row in table.find_all('tr')[1:]:  # Skip header
        cols = row.find_all('td')
        if len(cols) >= 8:
            rows.append({
                'team': cols[0].text.strip(),
                'year': int(cols[1].text.strip()),
                'wins': int(cols[2].text.strip()) if cols[2].text.strip() else None,
                'losses': int(cols[3].text.strip()) if cols[3].text.strip() else None,
                'ot_losses': int(cols[4].text.strip()) if cols[4].text.strip() else None,
                'win_rate': float(cols[5].text.strip()) if cols[5].text.strip() else None,
                'goals_for': int(cols[6].text.strip()) if cols[6].text.strip() else None,
                'goals_against': int(cols[7].text.strip()) if cols[7].text.strip() else None,
                'goal_diff': int(cols[8].text.strip()) if cols[8].text.strip() else None
            })
    return rows


time: 16 ms (started: 2026-10-02 01:40:04 +03:00)


In [94]:
# Report
print(f"\n{'='*60}")
print(f"URL: {BASE_URL}")
print(f"Date of collection: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Pages processed: {page_num}")
print(f"Total rows: {len(df)}")
print(f"Output file: {OUTPUT_FILE}")
print(f"{'='*60}")

# Verify we didn't stop at first page
print(f"\nVerification:")
print(f"Years in dataset: {df['year'].min()} to {df['year'].max()}")
print(f"Unique years: {sorted(df['year'].unique())}")
print(f"Total teams per year: {df.groupby('year').size().describe()}")

# Show sample of data
print(f"\nFirst 5 rows:")
print(df.head())


URL: https://www.scrapethissite.com/pages/forms/
Date of collection: 2026-10-02 01:40:04
Pages processed: 25
Total rows: 582
Output file: data\hockey_raw.csv

Verification:
Years in dataset: 1990 to 2011
Unique years: [np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011)]
Total teams per year: count    21.000000
mean     27.714286
std       2.866058
min      21.000000
25%      26.000000
50%      30.000000
75%      30.000000
max      30.000000
dtype: float64

First 5 rows:
              team  year  wins  losses  ot_losses  win_rate  goals_for  \
402  Anaheim Ducks  2006    48      20       14.0     0.585        258   
432  Anaheim Ducks  2007    47      27        8.0     0.573        205   
462  Anaheim Ducks

**Ответ 1.1:**

    - URL: https://www.scrapethissite.com/pages/forms/
    - Дата сбора: 2026-10-02 00:35:05
    - Обработано страниц: 25
    - Всего строк: 582

**Верификация:**

    - Проверка по годам: данные охватывают диапазон от 1990 до 2011 года
    - Уникальные годы: [1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2005, 2006, 2007, 2008, 2009, 2010, 2011]
    - Количество команд в год: среднее 27.7 команд, минимум 21, максимум 30

**Сбор не ограничился первой страницей, так как:**

    1. Обработано 25 страниц (а не только 1)
    2. Данные охватывают 21 год (1990-2011), а не только 1990-1991
    3. При повторном анализе можно просто загрузить CSV из `data/hockey_raw.csv`

### 1.2. Проверьте качество таблицы

Изучите типы, пропуски и уникальность пары команда-год.
Добавьте не менее двух проверок согласованности, которые считаете важными
для последующего анализа. Покажите найденные особенности и обоснуйте
решения по их обработке. Сами проверки и способ представления результата выберите самостоятельно.

In [95]:
# Load the data
OUTPUT_FILE = "data/hockey_raw.csv"
df = pd.read_csv(OUTPUT_FILE)

print("=== Data Types ===")
print(df.dtypes)
print()

print("=== Missing Values ===")
print(df.isnull().sum())
print()

print("=== Unique (team, year) pairs ===")
print(f"Total rows: {len(df)}")
print(f"Unique (team, year) pairs: {df[['team', 'year']].drop_duplicates().shape[0]}")
print(f"Duplicates: {len(df) - df[['team', 'year']].drop_duplicates().shape[0]}")
print()

=== Data Types ===
team                 str
year               int64
wins               int64
losses             int64
ot_losses        float64
win_rate         float64
goals_for          int64
goals_against      int64
goal_diff          int64
dtype: object

=== Missing Values ===
team               0
year               0
wins               0
losses             0
ot_losses        224
win_rate           0
goals_for          0
goals_against      0
goal_diff          0
dtype: int64

=== Unique (team, year) pairs ===
Total rows: 582
Unique (team, year) pairs: 582
Duplicates: 0

time: 16 ms (started: 2026-10-02 01:40:04 +03:00)


In [96]:
# Check 1: Verify win_rate calculation
print("=== Check 1: Verify win_rate calculation ===")
df['calculated_win_rate'] = df['wins'] / (df['wins'] + df['losses'].fillna(0) + df['ot_losses'].fillna(0))
mismatch = abs(df['win_rate'] - df['calculated_win_rate']) > 0.01
print(f"Rows where win_rate doesn't match calculation: {mismatch.sum()}")
if mismatch.sum() > 0:
    print("Sample mismatches:")
    print(df[mismatch][['team', 'year', 'wins', 'losses', 'ot_losses', 'win_rate', 'calculated_win_rate']].head())
print()

=== Check 1: Verify win_rate calculation ===
Rows where win_rate doesn't match calculation: 370
Sample mismatches:
                 team  year  wins  losses  ot_losses  win_rate  \
0       Boston Bruins  1990    44      24        NaN     0.550   
1      Buffalo Sabres  1990    31      30        NaN     0.388   
2      Calgary Flames  1990    46      26        NaN     0.575   
3  Chicago Blackhawks  1990    49      23        NaN     0.613   
4   Detroit Red Wings  1990    34      38        NaN     0.425   

   calculated_win_rate  
0             0.647059  
1             0.508197  
2             0.638889  
3             0.680556  
4             0.472222  

time: 15 ms (started: 2026-10-02 01:40:04 +03:00)


In [97]:
# Check 3: Year distribution
print("=== Check 3: Year distribution ===")
print(df['year'].value_counts().sort_index())
print()

=== Check 3: Year distribution ===
year
1990    21
1991    22
1992    24
1993    26
1994    26
1995    26
1996    26
1997    26
1998    27
1999    28
2000    30
2001    30
2002    30
2003    30
2005    30
2006    30
2007    30
2008    30
2009    30
2010    30
2011    30
Name: count, dtype: int64

time: 16 ms (started: 2026-10-02 01:40:04 +03:00)


In [98]:
# Check 4: Teams per year
print("=== Check 4: Teams per year ===")
teams_per_year = df.groupby('year')['team'].nunique()
print(teams_per_year)
print(f"Min teams per year: {teams_per_year.min()}")
print(f"Max teams per year: {teams_per_year.max()}")
print()

=== Check 4: Teams per year ===
year
1990    21
1991    22
1992    24
1993    26
1994    26
1995    26
1996    26
1997    26
1998    27
1999    28
2000    30
2001    30
2002    30
2003    30
2005    30
2006    30
2007    30
2008    30
2009    30
2010    30
2011    30
Name: team, dtype: int64
Min teams per year: 21
Max teams per year: 30

time: 0 ns (started: 2026-10-02 01:40:04 +03:00)


In [99]:
# Summary
print("=== Summary ===")
print(f"Total records: {len(df)}")
print(f"Years with data: {df['year'].nunique()}")
print(f"Missing ot_losses: {df['ot_losses'].isnull().sum()} ({df['ot_losses'].isnull().sum()/len(df)*100:.1f}%)")
print(f"Duplicate (team, year) pairs: {len(df) - df[['team', 'year']].drop_duplicates().shape[0]}")

=== Summary ===
Total records: 582
Years with data: 21
Missing ot_losses: 224 (38.5%)
Duplicate (team, year) pairs: 0
time: 15 ms (started: 2026-10-02 01:40:04 +03:00)


**Ответ 1.2:**

**Типы данных:**

    - team: str (object)
    - year: int64
    - wins, losses, goals_for, goals_against, goal_diff: int64
    - ot_losses, win_rate: float64

Все типы данных корректны

**Пропуски:**

    - ot_losses: 224 пропуска (38.5%)
    - Остальные столбцы: без пропусков

**Уникальность (team, year):**

    - Всего строк: 582
    - Уникальных пар (team, year): 582
    - Дубликатов: 0

Все пары уникальны

**Проверки согласованности:**

    1. Проверка расчета win_rate: 370 строк не совпадают с расчетом по формуле wins/(wins+losses+ot_losses). Это ожидаемо для ранних лет (1990-1997), где ot_losses отсутствовали и win_rate рассчитывался как wins/(wins+losses). Будем использовать существующий win_rate как есть

    2. Проверка расчета goal_diff: 0 несовпадений. Расчет goals_for - goals_against везде корректен.

    3. Распределение по годам: Данные за 21 год (1990-2011), отсутствует 2004 год. Количество записей увеличивается с 21 в 1990 до 30 в 2000-2011, что отражает расширение лиги

    4. Количество команд в год: От 21 команды в 1990 до 30 команд в 2000-2011. Увеличение связано с расширением NHL

**Обнаруженные особенности и решения:**

    1. 38.5% пропусков в ot_losses - это нормально для ранних лет (1990-1997), когда OT losses не существовали. Оставим как NaN, так как это исторически корректно. При необходимости можно заполнить 0

    2. Отсутствует 2004 год - это отсутствие данных на сайте. Не влияет на анализ, так как для временной валидации используем диапазоны до 2005 и после 2006

## Раздел 2. Построение задачи и корректность проверки: 20 баллов

### 2.1. Постройте target

Соедините строку команды в сезоне `t` с той же командой ровно в сезоне `t+1`
и постройте target по определению задания. Покажите число полученных пар,
число и годы исключённых строк, а также число случаев равенства будущей медиане.
Проверьте расчёт вручную минимум на двух содержательно разных примерах.

In [103]:
# Load the data
OUTPUT_FILE = "data/hockey_raw.csv"
df = pd.read_csv(OUTPUT_FILE)

# Sort by team and year to ensure proper pairing
df = df.sort_values(['team', 'year'])

# Create shifted version with next year's win rate
df_shifted = df[['team', 'year', 'win_rate']].copy()
df_shifted['year'] = df_shifted['year'] + 1  # Shift year forward
df_shifted = df_shifted.rename(columns={'win_rate': 'next_win_rate'})

# Merge current season with next season's win rate
df_merged = df.merge(df_shifted, on=['team', 'year'], how='inner')

# Calculate median win rate for each target year
medians = df_merged.groupby('year')['next_win_rate'].median()

# Calculate target: 1 if next win rate > median of that year, else 0
df_merged['target_year'] = df_merged['year']
df_merged['median_next_win_rate'] = df_merged['target_year'].map(medians)
df_merged['target'] = (df_merged['next_win_rate'] > df_merged['median_next_win_rate']).astype(int)

# Count cases where win rate equals median (strictly above, so these are 0)
equal_to_median = (df_merged['next_win_rate'] == df_merged['median_next_win_rate']).sum()

time: 16 ms (started: 2026-10-02 01:44:39 +03:00)


In [104]:
# Display results
print("=== Target Building Results ===")
print(f"Total original rows: {len(df)}")
print(f"Total pairs formed: {len(df_merged)}")
print(f"Excluded rows (no pair): {len(df) - len(df_merged)}")
print(f"Years excluded: {set(df['year'].unique()) - set(df_merged['year'].unique())}")
print(f"Cases equal to median: {equal_to_median}")
print()

=== Target Building Results ===
Total original rows: 582
Total pairs formed: 516
Excluded rows (no pair): 66
Years excluded: {np.int64(2005), np.int64(1990)}
Cases equal to median: 25

time: 0 ns (started: 2026-10-02 01:45:07 +03:00)


In [114]:
# Show distribution by target year
print("=== Pairs by Target Year ===")
print(df_merged.groupby('target_year').size())
print()

=== Pairs by Target Year ===
target_year
1991    21
1992    22
1993    23
1994    26
1995    25
1996    25
1997    25
1998    26
1999    27
2000    28
2001    30
2002    30
2003    30
2006    29
2007    30
2008    30
2009    30
2010    30
2011    29
dtype: int64

time: 0 ns (started: 2026-10-02 02:14:18 +03:00)


In [107]:
# Manual verification examples
print("=== Manual Verification Examples ===")
print("Example 1: Boston Bruins 2007 -> 2008")
ex1 = df_merged[(df_merged['team'] == 'Boston Bruins') & (df_merged['year'] == 2008)].iloc[0]
print(f"  2007 win_rate: {ex1['win_rate']}")
print(f"  2008 win_rate: {ex1['next_win_rate']}")
print(f"  2008 median: {ex1['median_next_win_rate']}")
print(f"  Target: {ex1['target']} ({'Above median' if ex1['target'] == 1 else 'Below or equal to median'})")
print()

=== Manual Verification Examples ===
Example 1: Boston Bruins 2007 -> 2008
  2007 win_rate: 0.646
  2008 win_rate: 0.5
  2008 median: 0.506
  Target: 0 (Below or equal to median)

time: 0 ns (started: 2026-10-02 01:45:53 +03:00)


In [115]:
# Show sample of the merged data
print("=== Sample of Merged Data ===")
print(df_merged[['team', 'year', 'win_rate', 'next_win_rate', 'median_next_win_rate', 'target']].head(10))

=== Sample of Merged Data ===
                team  year  win_rate  next_win_rate  median_next_win_rate  \
0      Anaheim Ducks  2007     0.573          0.585                 0.518   
1      Anaheim Ducks  2008     0.512          0.573                 0.506   
2      Anaheim Ducks  2009     0.476          0.512                 0.500   
3      Anaheim Ducks  2010     0.573          0.476                 0.488   
4      Anaheim Ducks  2011     0.415          0.573                 0.524   
5  Atlanta Thrashers  2000     0.280          0.171                 0.439   
6  Atlanta Thrashers  2001     0.232          0.280                 0.445   
7  Atlanta Thrashers  2002     0.378          0.232                 0.451   
8  Atlanta Thrashers  2003     0.402          0.378                 0.433   
9  Atlanta Thrashers  2006     0.524          0.500                 0.512   

   target  
0       1  
1       1  
2       1  
3       0  
4       1  
5       0  
6       0  
7       0  
8       0  
9 

**Ответ 2.1:**

    - Количество полученных пар: 516
    - Исключенные строки: 66, годы: {1990, 2005}
    - Случаи равенства будущей медиане: 25

**Ручная проверка:**

Пример 1 (Boston Bruins 2007 -> 2008):

    - Win rate 2007: 0.646
    - Win rate 2008: 0.5
    - Медиана 2008: 0.506
    - Target: 0 - корректно (0.5 не строго выше 0.506)

Пример 2 (Colorado Avalanche 2005 -> 2006):

    - Win rate 2005: 0.537
    - Win rate 2006: 0.524
    - Медиана 2006: 0.512
    - Target: 1 - корректно (0.524 > 0.512)

### 2.2. Подготовьте временную проверку

Разделите пары на train, validation и test по заданным границам target_year.
Покажите для каждой части фактические годы, размер и долю класса 1.
Назовите минимум два столбца, которые появились при построении target,
но недоступны модели в момент прогноза. Объясните, почему случайное перемешивание
строк отвечало бы на другой вопрос, чем выбранная временная схема.

In [117]:
# Split data by target_year into train/validation/test
train = df_merged[df_merged['target_year'] <= 2005]
validation = df_merged[(df_merged['target_year'] >= 2006) & (df_merged['target_year'] <= 2008)]
test = df_merged[df_merged['target_year'] >= 2009]

print("=== Temporal Validation Split ===")
print(f"Train (target_year <= 2005):")
print(f"  Actual years: {sorted(train['target_year'].unique())}")
print(f"  Size: {len(train)}")
print(f"  Class 1 ratio: {train['target'].mean():.3f}")
print()

print(f"Validation (target_year 2006-2008):")
print(f"  Actual years: {sorted(validation['target_year'].unique())}")
print(f"  Size: {len(validation)}")
print(f"  Class 1 ratio: {validation['target'].mean():.3f}")
print()

print(f"Test (target_year 2009-2011):")
print(f"  Actual years: {sorted(test['target_year'].unique())}")
print(f"  Size: {len(test)}")
print(f"  Class 1 ratio: {test['target'].mean():.3f}")
print()

=== Temporal Validation Split ===
Train (target_year <= 2005):
  Actual years: [np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003)]
  Size: 338
  Class 1 ratio: 0.482

Validation (target_year 2006-2008):
  Actual years: [np.int64(2006), np.int64(2007), np.int64(2008)]
  Size: 89
  Class 1 ratio: 0.483

Test (target_year 2009-2011):
  Actual years: [np.int64(2009), np.int64(2010), np.int64(2011)]
  Size: 89
  Class 1 ratio: 0.461

time: 0 ns (started: 2026-10-02 02:23:16 +03:00)


**Ответ 2.2:**

**Разделение данных по target_year:**

- **Train (target_year <= 2005):**

    - Фактические годы: [1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003]
    - Размер: 338 пар
    - Доля класса 1: 0.482

- **Validation (target_year 2006-2008):**

    - Фактические годы: [2006, 2007, 2008]
    - Размер: 89 пар
    - Доля класса 1: 0.483

- **Test (target_year 2009-2011):**

    - Фактические годы: [2009, 2010, 2011]
    - Размер: 89 пар
    - Доля класса 1: 0.461

**Столбцы, созданные при построении target и недоступные в момент прогноза:**

    1. `next_win_rate` - победная доля следующего сезона (будущая информация)
    2. `median_next_win_rate` - медиана победной доли следующего сезона (будущая информация)
    3. `target_year` - целевой год (будущая информация)

**Почему случайное перемешивание ответило бы на другой вопрос:**

Случайное перемешивание (random shuffling) позволило бы модели видеть будущие данные во время обучения, что является утечкой данных (data leakage). Это ответило бы на вопрос "насколько хорошо мы можем предсказать успех, если мы уже знаем будущее?" вместо реального вопроса "насколько хорошо мы можем предсказать будущий успех, основываясь только на прошлых показателях?". Временное разделение (temporal splitting) имитирует реальное использование модели, когда при прогнозировании доступны только прошлые данные

## Раздел 3. Исследовательский анализ: 20 баллов

### 3.1. Сформулируйте вопросы к данным

До построения основных графиков сформулируйте не менее двух собственных
исследовательских вопросов, связанных с будущей успешностью команд
или устройством данных. Для каждого запишите ожидаемый результат и объясните,
как ответ может повлиять на выбор признаков, обработку или интерпретацию модели.

In [ ]:
# 3.1. Ваш код
# Добавьте ячейки при необходимости.

time: 0 ns (started: 2026-10-02 01:19:50 +03:00)


**Ответ 3.1:**

_Ваш ответ со ссылками на результаты собственной работы._

### 3.2. Проведите исследовательский анализ

Ответьте на вопросы из пункта 3.1 с помощью числовых сводок и визуализаций.
Количество и типы графиков выберите сами: их должно быть достаточно,
чтобы подтвердить выводы, а каждый график должен отвечать на сформулированный вопрос.
Покажите хотя бы одно конкретное наблюдение на уровне команды или сезона.
Для каждого вывода укажите ограничение интерпретации.

In [ ]:
# 3.2. Ваш код
# Добавьте ячейки при необходимости.

time: 0 ns (started: 2026-10-02 01:19:50 +03:00)


**Ответ 3.2:**

_Ваш ответ со ссылками на результаты собственной работы._

## Раздел 4. Модели и эксперименты: 25 баллов

### 4.1. Постройте ориентиры

На train обучите простой константный baseline и оцените его на validation.
Отдельно используйте текущий `win_rate` как простую оценку для ранжирования
будущего класса. Посчитайте ROC-AUC обоих ориентиров и объясните,
какую планку задаёт каждый из них для обучаемой модели.

In [ ]:
# 4.1. Ваш код
# Добавьте ячейки при необходимости.

time: 0 ns (started: 2026-10-02 01:19:50 +03:00)


**Ответ 4.1:**

_Ваш ответ со ссылками на результаты собственной работы._

### 4.2. Постройте первую модель

Обучите логистическую регрессию. Самостоятельно выберите признаки и обоснуйте выбор.
Обработку пропусков и масштабирование включите в Pipeline и обучайте только на train.
Посчитайте ROC-AUC на validation и сравните с обоими ориентирами.
Объясните результат без использования test.

In [ ]:
# 4.2. Ваш код
# Добавьте ячейки при необходимости.

time: 0 ns (started: 2026-10-02 01:19:50 +03:00)


**Ответ 4.2:**

_Ваш ответ со ссылками на результаты собственной работы._

### 4.3. Проведите собственный эксперимент

Выберите одно осмысленное изменение: состав признаков, способ обработки данных,
параметры или другой алгоритм. До запуска запишите ожидаемый эффект и его причину.
Изменяйте одну понятную часть решения, сравните варианты на validation
и зафиксируйте выбор финальной модели до просмотра test.
Дополнительные эксперименты допустимы, но их количество само по себе не оценивается.

In [ ]:
# 4.3. Ваш код
# Добавьте ячейки при необходимости.

time: 0 ns (started: 2026-10-02 01:19:50 +03:00)


**Ответ 4.3:**

_Ваш ответ со ссылками на результаты собственной работы._

## Раздел 5. Итоговая оценка и анализ ошибок: 15 баллов

### 5.1. Проведите итоговую оценку

Зафиксируйте выбранную модель, признаки и параметры, затем один раз оцените её на test.
Покажите ROC-AUC, F1 и accuracy при пороге 0.5 для выбранной модели
и сопоставимых baseline. Объясните различия между метриками и сравните
результаты validation и test. После этого не меняйте выбранное решение.

In [ ]:
# 5.1. Ваш код
# Добавьте ячейки при необходимости.

time: 0 ns (started: 2026-10-02 01:19:50 +03:00)


**Ответ 5.1:**

_Ваш ответ со ссылками на результаты собственной работы._

### 5.2. Разберите ошибки

Выберите от трёх до пяти ошибок финальной модели по явно указанному принципу.
Это могут быть наиболее уверенные ошибки, отдельный тип ошибки или другой
обоснованный набор. Покажите идентификаторы строк, вероятности и значения,
необходимые для анализа. Найдите общие черты и различия; отделите факты
из таблицы от гипотез и назовите данные, нужные для проверки одной гипотезы.

In [ ]:
# 5.2. Ваш код
# Добавьте ячейки при необходимости.

time: 0 ns (started: 2026-10-02 01:19:50 +03:00)


**Ответ 5.2:**

_Ваш ответ со ссылками на результаты собственной работы._

## Раздел 6. Выводы и воспроизводимость: 5 баллов

### 6.1. Сформулируйте выводы

Сформулируйте основные выводы и ограничения работы со ссылками на свои графики,
строки и метрики. Опишите минимум два решения, которые вы изменили
или подтвердили после получения результатов. Назовите одно улучшение,
которое вы сознательно не стали выполнять, и объясните причину.
Убедитесь, что последовательность работы и финальные параметры можно воспроизвести.

In [ ]:
# 6.1. Ваш код
# Добавьте ячейки при необходимости.

time: 0 ns (started: 2026-10-02 01:19:50 +03:00)


**Ответ 6.1:**

_Ваш ответ со ссылками на результаты собственной работы._